# Backtest Report + LLM Workflow

You've just finished running all your backtests and have them saved to disk.
Now what? This notebook walks through the **complete workflow** — from loading
results, to opening the interactive dashboard, to connecting an LLM via the
built-in MCP server for AI-powered analysis.

## The Big Picture

```
┌─────────────────┐     ┌──────────────────────┐     ┌─────────────────────┐
│  Run Backtests  │ ──▶ │  Open Dashboard      │ ──▶ │  Start MCP Server   │
│  (saved to disk)│     │  (report.show())     │     │  (iaf mcp -d ...)   │
└─────────────────┘     └──────────────────────┘     └─────────────────────┘
                                  │                            │
                                  ▼                            ▼
                        ┌──────────────────────┐     ┌─────────────────────┐
                        │  Browse charts,      │     │  LLM queries data,  │
                        │  compare strategies, │ ◀─▶ │  creates notes,     │
                        │  capture snapshots   │     │  classifies strats  │
                        └──────────────────────┘     └─────────────────────┘
                                  │
                                  ▼
                        ┌──────────────────────┐
                        │  Apply Selections,   │
                        │  Export final report  │
                        └──────────────────────┘
```

## Step 1 — Load Your Backtest Results

After running your backtests, each strategy run is stored in its own folder
inside a **batch directory**. The `BacktestReport.open()` method scans the
directory and loads every valid backtest it finds.

In [ ]:
import os
from investing_algorithm_framework import BacktestReport, recalculate_backtests

# Point to the directory that contains all your backtest folders
batch_dir = "your_batch_directory"

# Load every backtest found in the directory
report = BacktestReport.open(directory_path=batch_dir)

# Recalculate all metrics (ensures everything is up-to-date)
recalculate_backtests(report.backtests)

print(f"Loaded {len(report.backtests)} strategies")
for bt in report.backtests:
    print(f"  • {bt.algorithm_id}")

## Step 1b — Tag Your Backtests

Tags let you label and group strategies — for example by batch name, experiment,
or configuration variant. Tags are **persisted** in a `tag.json` file inside
each backtest directory, so they survive across subsequent loads.

### How tags get assigned

There are three ways a backtest gets a tag:

**1. During the backtest run** — Set `tag` in the strategy metadata:
```python
strategy = TradingStrategy(
    algorithm_id="momentum_v3",
    metadata={"tag": "experiment_A"},
    ...
)
```
The tag is automatically extracted and saved with the backtest.

**2. After the fact with `retag_backtests()`** — Retag all backtests in a
directory (or filter by strategy_id):
```python
from investing_algorithm_framework import retag_backtests

# Tag everything in a directory
retag_backtests("experiment_A", directory_path="my_batch")

# Tag only a specific strategy
retag_backtests("experiment_B", directory_path="my_batch", strategy_id="momentum_v3")
```

**3. Multi-directory loading** — When you load from multiple directories,
each directory name is used as the default tag (unless a persisted tag exists):
```python
report = BacktestReport.open(directory_path=["batch_A", "batch_B"])
# Strategies from batch_A get tag "batch_A", etc.
```

### What tags do in the dashboard

- **Filter bar** — When 2+ tags exist, a chip-based filter bar appears at the top
- **Tag badges** — Every strategy name shows its tag badge in all tables
- **Collapsible sidebar groups** — Strategies are grouped by tag with collapsible headers
- **MCP server** — The `list_strategies` and `get_strategy_details` tools include tag info

In [ ]:
from investing_algorithm_framework import retag_backtests

# Retag all backtests in a directory
count = retag_backtests("experiment_A", directory_path=batch_dir)
print(f"Retagged {count} backtests with tag 'experiment_A'")

# Retag a single strategy by its algorithm_id
count = retag_backtests(
    "experiment_B",
    directory_path=batch_dir,
    strategy_id="momentum_v3"
)
print(f"Retagged {count} backtests with tag 'experiment_B'")

# Reload to see the tags
report = BacktestReport.open(directory_path=batch_dir)
for bt in report.backtests:
    print(f"  • {bt.algorithm_id}: tag={bt.tag}")

### Multi-directory loading

You can also load backtests from **multiple directories** at once by passing a list.
Each directory name becomes the default tag for its strategies (unless they already
have a persisted tag).

In [ ]:
# Load from multiple directories — each dir name becomes the tag
report = BacktestReport.open(directory_path=["batch_A", "batch_B"])

for bt in report.backtests:
    print(f"  • {bt.algorithm_id}: tag={bt.tag}")

# Open the dashboard — strategies will be grouped by tag in the sidebar
report.show()

## Step 2 — Open the Interactive Dashboard

`report.show()` generates a full HTML dashboard and opens it in your browser.
When run inside a Jupyter notebook, it also renders inline.

The dashboard gives you:
- **Overview** — KPI cards, equity curves, monthly/yearly returns for all strategies
- **Per-strategy pages** — Trading activity, drawdown, rolling Sharpe, return scenarios
- **Compare page** — Side-by-side strategy comparison
- **Report Builder** (right panel) — Notes, snapshots, selections, and export

In [ ]:
# Opens the dashboard in your browser (and renders inline in Jupyter)
report.show()

## Step 3 — Start the MCP Server (Connect Your LLM)

This is where the magic happens. The framework ships with a built-in
**MCP (Model Context Protocol) server** that exposes all your backtest data
as tools an LLM can call. This means GitHub Copilot, Claude, ChatGPT, or any
MCP-compatible agent can directly query your strategies, metrics, trades,
equity curves, and more — and even create analysis notes on the dashboard.

### Start the server from your terminal

```bash
# Single directory
iaf mcp -d your_batch_directory

# Multiple directories — repeat -d for each
iaf mcp -d batch_one -d batch_two -d batch_three
```

Or if you haven't installed the CLI:

```bash
python -m investing_algorithm_framework.cli.mcp_server -d your_batch_directory

# Multiple directories
python -m investing_algorithm_framework.cli.mcp_server -d batch_one -d batch_two
```

### Configure VS Code to use it

Add this to your `.vscode/settings.json` (or the MCP config UI):

```json
{
  "mcp": {
    "servers": {
      "backtest-analysis": {
        "command": "iaf",
        "args": ["mcp", "-d", "batch_one", "-d", "batch_two"]
      }
    }
  }
}
```

Once running, your LLM in Copilot Chat (or any MCP client) automatically
discovers **25 tools** it can call.

## Step 4 — The LLM Toolkit (25 MCP Tools)

When the MCP server starts, the LLM gets access to these tool categories:

### Data Exploration
| Tool | What it does | Supports `strategy_ids`? | Supports `tag`? |
|------|-------------|:---:|:---:|
| `list_strategies` | List all strategies with key metrics | ✅ | ✅ |
| `get_strategy_details` | Deep-dive into one strategy (all metrics, parameters) | — | — |
| `rank_strategies` | Rank by any metric (CAGR, Sharpe, max drawdown, etc.) | ✅ | ✅ |
| `compare_strategies` | Side-by-side comparison of 2 strategies | — | — |
| `get_full_analysis` | Complete data dump for AI analysis | ✅ | ✅ |
| `get_trading_activity` | Trading Activity table — all 12 trading metrics | ✅ | ✅ |

### Time Series (all support multi-strategy stacking + window filter)
| Tool | What it does | Supports `strategy_ids`? | Supports `tag`? |
|------|-------------|:---:|:---:|
| `get_equity_curve` | Equity curve (portfolio value over time) | ✅ | ✅ |
| `get_drawdown_series` | Drawdown over time | ✅ | ✅ |
| `get_monthly_returns` | Monthly return heatmap grid | ✅ | ✅ |
| `get_yearly_returns` | Yearly return summary | ✅ | ✅ |
| `get_rolling_sharpe` | Rolling Sharpe ratio series | ✅ | ✅ |
| `get_correlation_matrix` | Strategy correlation matrix | ✅ | ✅ |
| `get_window_coverage` | Backtest window coverage info | ✅ | ✅ |
| `get_portfolio_snapshots` | Portfolio initial/final value & growth | ✅ | ✅ |

### Trades, Orders & Positions (single strategy)
| Tool | What it does |
|------|-------------|
| `get_trades` | All trades with entry/exit prices, P&L (top N by return magnitude) |
| `get_orders` | All orders with side, type, status, price, amount, fee, slippage |
| `get_positions` | All positions with symbol, amount, cost |
| `get_symbol_breakdown` | Per-symbol performance breakdown |
| `get_return_scenarios` | Best/worst case return scenarios |

**Multi-strategy usage:** Pass `strategy_ids: ["strat_A", "strat_B", "strat_C"]`
instead of `strategy_id` to get a stacked comparison. Add `window: "2024"`
to filter to a specific backtest window. Or use `tag: "experiment_A"`
to filter by batch tag.

### Notes & Report Building
| Tool | What it does |
|------|-------------|
| `create_note` | Create a markdown note with strategy selections |
| `list_notes` | List all existing notes |
| `get_note` | Read a note (shows snapshot IDs for embedding) |
| `update_note` | Update note content, selections, or embed snapshots |
| `delete_note` | Remove a note |
| `filter_strategies` | Filter strategies by metric conditions (supports `strategy_ids` + `tag` pre-filtering) |

## Step 5 — Ask the LLM to Analyze Your Strategies

With the dashboard open in your browser and the MCP server running, you talk
to the LLM in **Copilot Chat** (or any MCP-connected client). Here are example
prompts and what happens behind the scenes:

---

### Example 1: "Analyze all my strategies and rank them"

**You say:**
> "I have a batch of backtest results. Rank them by risk-adjusted return and
> tell me which ones are worth keeping."

**The LLM will:**
1. Call `list_strategies` → gets all strategy names + summary metrics
2. Call `rank_strategies` with metric `"sharpe_ratio"` → sorted ranking
3. Call `get_strategy_details` for the top 3 → deep dives into each
4. Call `get_drawdown_series` for the top 3 → checks risk profile
5. Synthesize everything into a written analysis

---

### Example 2: "Create an analysis note with your findings"

**You say:**
> "Write up your analysis as a note. Mark the best strategies as keep,
> mediocre ones as maybe, and bad ones as reject."

**The LLM will:**
1. Call `create_note` with:
   - `title`: "Strategy Ranking Analysis"
   - `markdown`: Full write-up with headers, tables, conclusions
   - `selections`: `{"keep": ["strat_A", "strat_B"], "maybe": ["strat_C"], "reject": ["strat_D", "strat_E"]}`

**On your dashboard:** The note instantly appears in the Report Builder panel.
You can read it, edit it, and click **"Apply Selection"** to filter the
dashboard to only show the strategies the LLM recommended.

---

### Example 3: "Embed the equity curve in the note"

If you've captured chart snapshots in the dashboard, the LLM can embed them:

**You say:**
> "Get my current note and embed the snapshots where they make sense."

**The LLM will:**
1. Call `get_note` → sees the note content + a table of available snapshots with IDs
2. Call `update_note` → inserts `![[snap:1]]`, `![[snap:2]]` etc. at relevant
   positions in the markdown

The snapshots render inline in the note — charts, tables, whatever you captured.

## Step 6 — The Iterative Workflow

The real power is the loop between **you**, the **dashboard**, and the **LLM**.
Here's how a typical session looks:

```
┌─ YOU (in Copilot Chat) ──────────────────────────────────────────────┐
│                                                                       │
│  "Rank all strategies by Sharpe ratio"                                │
│       │                                                               │
│       ▼                                                               │
│  LLM calls: list_strategies → rank_strategies → get_strategy_details │
│  LLM responds with a ranking + analysis                              │
│       │                                                               │
│       ▼                                                               │
│  "Create a note with your analysis, classify keep/maybe/reject"      │
│       │                                                               │
│       ▼                                                               │
│  LLM calls: create_note(title, markdown, selections)                 │
│  → Note appears on the dashboard Report Builder panel                │
│                                                                       │
├─ YOU (in the Dashboard) ─────────────────────────────────────────────┤
│                                                                       │
│  • Read the note, tweak selections if you disagree                   │
│  • Click "Apply Selection" → dashboard filters to keep/maybe only    │
│  • Browse the filtered charts, capture snapshots of key charts       │
│  • Expand the panel to half-screen for easier reading                │
│                                                                       │
├─ YOU (back in Copilot Chat) ─────────────────────────────────────────┤
│                                                                       │
│  "Why does Strategy X have such high drawdown in Q3?"                │
│       │                                                               │
│       ▼                                                               │
│  LLM calls: get_drawdown_series → get_trades (filtered by date)     │
│  LLM explains the specific trades that caused the drawdown           │
│       │                                                               │
│       ▼                                                               │
│  "Update the note with this finding and embed the snapshots"         │
│       │                                                               │
│       ▼                                                               │
│  LLM calls: get_note → update_note (adds ![[snap:1]] inline)        │
│  → Dashboard note updates with embedded chart images                 │
│                                                                       │
└───────────────────────────────────────────────────────────────────────┘
```

The notes are **synced** — what the LLM writes via MCP appears in the dashboard,
and snapshots you capture in the dashboard are available to the LLM via `get_note`.

## Step 7 — Notes as Self-Contained Analysis Documents

Each note on the dashboard is a rich markdown document that bundles:

| Component | Description |
|-----------|-------------|
| **Title** | Note heading (e.g., "Q1 Strategy Selection") |
| **Markdown** | Full analysis text — headers, tables, bullet points, code blocks |
| **Selections** | Strategy classifications: `keep` / `maybe` / `reject` |
| **Snapshots** | Captured charts/tables from the dashboard, embeddable via `![[snap:ID]]` |

### The `![[snap:ID]]` Embed Syntax

When you capture a chart snapshot on the dashboard, it gets an ID. You (or the LLM)
can embed it anywhere in the markdown:

```markdown
## Equity Curve Analysis

The top 3 strategies show consistent upward trends:

![[snap:1]]

However, Strategy X shows a significant drawdown in Q3:

![[snap:2]]

This correlates with the high-volatility period we identified.
```

The dashboard renders these as inline images/tables within the note body.
You can also **drag and drop** snapshots directly onto the editor, or use
the **Insert** button on each snapshot card.

## Step 8 — Example Conversation with the LLM

Here's a realistic end-to-end conversation you'd have in Copilot Chat:

---

**You:** *"I just ran 30 strategy backtests. Give me a quick overview — which
ones have the best risk-adjusted returns?"*

**LLM:** *(calls `list_strategies`, then `rank_strategies` by sharpe_ratio)*

> Here are your top 5 strategies by Sharpe ratio:
>
> | Rank | Strategy | Sharpe | CAGR | Max DD |
> |------|----------|--------|------|--------|
> | 1 | momentum_v3 | 1.82 | 24.1% | -12.3% |
> | 2 | mean_rev_opt | 1.65 | 18.7% | -8.9% |
> | ... | ... | ... | ... | ... |
>
> Strategies momentum_v3 and mean_rev_opt stand out with high Sharpe ratios
> and controlled drawdowns...

---

**You:** *"Compare momentum_v3 and mean_rev_opt in detail"*

**LLM:** *(calls `compare_strategies`, `get_equity_curve` for both, `get_trades` for both)*

> Detailed comparison:
> - **momentum_v3** has higher returns but trades more frequently (142 vs 87 trades)
> - **mean_rev_opt** has lower drawdown and more consistent monthly returns...

---

**You:** *"Write this up as an analysis note. Keep both of those, mark the
bottom 10 as reject, rest as maybe."*

**LLM:** *(calls `create_note` with full markdown analysis + selections)*

> ✅ Created note "Strategy Selection Analysis" with 2 keep, 18 maybe, 10 reject.
> It's now visible in your dashboard Report Builder.

---

**You:** *"I captured some chart snapshots on the dashboard. Embed them in the note."*

**LLM:** *(calls `get_note` → sees snapshot IDs → calls `update_note` with `![[snap:ID]]`)*

> ✅ Updated the note — embedded 3 snapshots:
> - Snap 1 (Equity Curves) after the overview section
> - Snap 2 (Drawdown Chart) after the risk section
> - Snap 3 (Monthly Returns) in the appendix

## Step 9 — Apply Selections & Export

Once your notes are ready (whether written by you, the LLM, or both):

### On the Dashboard

1. **Apply Selection** — Click the button on a note to filter the dashboard.
   Only `keep` and `maybe` strategies will appear in all charts and tables.
   `reject` strategies are hidden. This lets you focus on your shortlist.

2. **Expand Panel** — Click the expand button (↗) in the Report Builder header
   to widen the panel to half the screen for easier reading/editing.

3. **Capture More Snapshots** — With the filtered view, capture charts that
   show only your shortlisted strategies. These give cleaner visuals for your
   final report.

### Export Options

Switch to the **Export** tab in the Report Builder:

- **📄 Copy as Markdown** — Copies all notes + embedded snapshots as markdown
  to your clipboard. Paste into Notion, Confluence, GitHub, etc.

- **💾 Save HTML Report** — Saves a self-contained HTML file with all data,
  charts, and notes baked in. Share with your team or archive it.

- **🤖 Copy All Data for AI Analysis** — Exports the full dataset as
  structured text. Useful for pasting into ChatGPT/Claude if you want
  a separate AI analysis outside of the MCP workflow.

## Step 10 — You Can Also Save / Reload Programmatically

The report and note state are persisted. The dashboard saves notes into the
HTML itself (via the Save button), and the MCP server stores them in a
`.analysis_notes.json` file in your batch directory. Both survive restarts.

In [ ]:
# Save the report as a standalone HTML file
report.save("my_analysis_report.html")
print("Report saved to my_analysis_report.html")

## TL;DR — The Complete Workflow

```python
from investing_algorithm_framework import BacktestReport, recalculate_backtests

# 1. Load all backtests
report = BacktestReport.open(directory_path="my_batch")
recalculate_backtests(report.backtests)

# 2. Open the dashboard
report.show()
```

```bash
# 3. Start the MCP server (in a separate terminal)
iaf mcp -d my_batch

# Or load multiple batch directories at once
iaf mcp -d batch_one -d batch_two
```

```
# 4. In Copilot Chat (or any MCP client):
"Analyze my strategies, rank by Sharpe, create a note with keep/maybe/reject"
"Embed the snapshots I captured in the note"
"Why does strategy X underperform in Q3?"

# 5. On the dashboard:
- Read the LLM's notes
- Click "Apply Selection" to filter
- Capture more snapshots
- Export the final report
```

That's it — your backtests, a visual dashboard, and an LLM that can read
the data and write analysis notes, all connected.